# Licenciatura en Inteligencia de Negocios  
## Curso: CD3002C.601 — Evidencia Clase 7

### Instituto Tecnológico y de Estudios Superiores de Monterrey  
**Campus:** Santa Fe

---

### Información General del Curso

- **Licenciatura:** Inteligencia de Negocios  
- **Concentración:** Inteligencia Artificial con Impacto Empresarial  
- **Semestre:** 6  

---
- **Actividad:** *Clusters como input para AI e interpretación*  

---

### Profesores del Curso

- **Profesora Titular:**  
  - Fabiola Celia Vásquez García  
- **Profesores de Apoyo:**  
  - Antonio Armando Ortiz Barrañón  
  - Arturo Tapia Sánchez  
  - Ricardo Cardoso Ortegón  
  - Erick Villalpando Terán  
  - Joel Castañeda Espinoza  
  - Sergio Haro Pérez  

---

### Equipo 5 — Miembros

| Matrícula     | Nombre                        |
|---------------|-------------------------------|
| A01663977     | Ludovic Delot Bravo           |
| A01784359     | Gonzalo González Méndez       |
| A01783752     | Anakarenina Serrano Ibarra    |
| A017716269    | Mónica Estrada Mondragón      |


# 1. Importación de librerías

In [157]:
def delot_setup_environment(silent: bool = True, save_summary: bool = False, summary_filename: str = "library_summary"):
    """
    delot_: Configura el entorno de trabajo. Instala, importa, valida y resume detalles del sistema y librerías clave.
    """
    # 🔧 Imports base
    import importlib as delot_importlib
    import subprocess as delot_subprocess
    import sys as delot_sys
    import time as delot_time
    import platform as delot_platform
    import os as delot_os
    from pathlib import Path as delot_Path
    from importlib.metadata import version as delot_version, PackageNotFoundError as delot_PackageNotFoundError
    from IPython.display import display as delot_display, Markdown as delot_Markdown
    from packaging import version as delot_pv

    # 🎨 Librerías visuales clave y TSNE (forzadas)
    import plotly.express as px
    import plotly.graph_objects as go
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.manifold import TSNE

    delot_start_time = delot_time.time()

    # 📦 Paquetes requeridos y versiones mínimas
    delot_pip_packages = {
        'pandas': '1.5.0',
        'numpy': '1.22.0',
        'matplotlib': '3.5.0',
        'seaborn': '0.11.2',
        'plotly': '5.5.0',
        'scikit-learn': '1.0.0',
        'scipy': '1.7.0'
    }

    # 🔍 Mapeo de importaciones
    delot_imports = {
        'pandas': 'pd',
        'numpy': 'np',
        'matplotlib.pyplot': 'plt',
        'seaborn': 'sns',
        'plotly.express': 'px',
        'plotly.graph_objects': 'go',
        'sklearn.preprocessing': ['StandardScaler'],
        'sklearn.cluster': ['KMeans', 'DBSCAN'],
        'sklearn.mixture': ['GaussianMixture'],
        'sklearn.metrics': ['silhouette_score', 'calinski_harabasz_score', 'davies_bouldin_score'],
        'sklearn.manifold': ['TSNE'],
        'scipy.stats': ['zscore']
    }

    # 🚀 Instalación automática de paquetes si faltan
    for delot_pkg in delot_pip_packages:
        delot_module_name = delot_pkg if delot_pkg != 'scikit-learn' else 'sklearn'
        try:
            delot_importlib.import_module(delot_module_name)
        except ImportError:
            delot_subprocess.run(
                [delot_sys.executable, "-m", "pip", "install", delot_pkg],
                stdout=delot_subprocess.DEVNULL if silent else None,
                stderr=delot_subprocess.DEVNULL if silent else None
            )

    # 🔧 Paquetes del sistema
    for delot_sys_pkg in ['psutil', 'py-cpuinfo', 'GPUtil']:
        try:
            delot_importlib.import_module(delot_sys_pkg.lower())
        except ImportError:
            delot_subprocess.run(
                [delot_sys.executable, "-m", "pip", "install", delot_sys_pkg],
                stdout=delot_subprocess.DEVNULL if silent else None,
                stderr=delot_subprocess.DEVNULL if silent else None
            )

    # ✅ Importaciones globales
    delot_globals = globals()
    delot_version_info = []
    delot_failed_imports = []

    for delot_lib, delot_alias_or_items in delot_imports.items():
        try:
            if isinstance(delot_alias_or_items, str):
                delot_globals[delot_alias_or_items] = delot_importlib.import_module(delot_lib)
            else:
                delot_mod = delot_importlib.import_module(delot_lib)
                for item in delot_alias_or_items:
                    delot_globals[item] = getattr(delot_mod, item)
        except Exception as delot_e:
            delot_failed_imports.append((delot_lib, str(delot_e)))

    # 🎨 Estilo visual por defecto
    try:
        sns.set(style="whitegrid")
        plt.rcParams["figure.figsize"] = (8, 5)
    except Exception:
        pass

    # 🔍 Validación de versiones
    for delot_pkg, delot_min_ver in delot_pip_packages.items():
        try:
            delot_inst_ver = delot_version(delot_pkg)
            delot_status = "✅" if delot_pv.parse(delot_inst_ver) >= delot_pv.parse(delot_min_ver) else "⚠️"
            delot_version_info.append((delot_pkg, delot_inst_ver, delot_min_ver, delot_status))
        except delot_PackageNotFoundError:
            delot_version_info.append((delot_pkg, "Not Installed", delot_min_ver, "❌"))

    # 🧪 Test funcional robusto
    try:
        import pandas as delot_pd
        import numpy as delot_np
        import matplotlib.pyplot as delot_plt

        delot_test_df = delot_pd.DataFrame(
            delot_np.random.rand(5, 5),
            columns=[f"col_{i}" for i in range(5)]
        )
        delot_test_df.plot(title="delot_ Functional Test Plot")
        delot_plt.close()
        delot_test_ok = "✅ Functional test passed"
    except Exception as delot_test_exception:
        delot_test_ok = f"❌ Functional test failed: {delot_test_exception}"

    # 💻 Info del sistema
    def delot_get_system_info():
        import psutil as delot_psutil
        import cpuinfo as delot_cpuinfo
        from shutil import disk_usage as delot_disk_usage
        try:
            import GPUtil as delot_GPUtil
            delot_gpu_list = delot_GPUtil.getGPUs()
            delot_gpu = delot_gpu_list[0].name if delot_gpu_list else "No GPU detected"
        except:
            delot_gpu = "GPUtil not available"

        delot_cpu = delot_cpuinfo.get_cpu_info()
        delot_ram = round(delot_psutil.virtual_memory().total / 1e+9, 2)
        delot_disk = delot_disk_usage("/")

        return {
            "OS": f"{delot_platform.system()} {delot_platform.release()} ({delot_platform.machine()})",
            "Processor": delot_cpu.get("brand_raw", "Unknown"),
            "Physical Cores": delot_psutil.cpu_count(logical=False),
            "Logical Cores": delot_psutil.cpu_count(logical=True),
            "RAM": f"{delot_ram} GB",
            "Disk Free": f"{round(delot_disk.free / 1e+9, 2)} GB",
            "Disk Total": f"{round(delot_disk.total / 1e+9, 2)} GB",
            "GPU": delot_gpu
        }

    delot_sysinfo = delot_get_system_info()
    delot_script_path = delot_Path().resolve()
    delot_env_name = delot_os.environ.get('CONDA_DEFAULT_ENV') or delot_os.environ.get('VIRTUAL_ENV') or "N/A"

    # 📋 Resumen Markdown
    delot_env_md = f"""
**📍 Path:** `{delot_script_path}`  
**🐍 Python:** `{delot_platform.python_version()}`  
**💻 Platform:** `{delot_sysinfo['OS']}`  
**📦 Virtual Env:** `{delot_env_name}`  
**🧪 Test:** {delot_test_ok}  
**⏱️ Time:** `{round(delot_time.time() - delot_start_time, 2)}s`
"""
    delot_sys_md = "### 🖥️ System Info\n" + "\n".join([f"- **{k}:** `{v}`" for k, v in delot_sysinfo.items()])
    delot_md_table = "| 📦 Package | 🔢 Installed | 🎯 Required | ✅ Status |\n|------------|--------------|--------------|------------|\n"
    for pkg, inst, req, stat in sorted(delot_version_info):
        delot_md_table += f"| `{pkg}` | `{inst}` | `{req}` | {stat} |\n"

    delot_display(delot_Markdown("## ⚙️ Environment Summary"))
    delot_display(delot_Markdown(delot_env_md))
    delot_display(delot_Markdown(delot_sys_md))
    delot_display(delot_Markdown("### 📋 Library Versions"))
    delot_display(delot_Markdown(delot_md_table))

    # 💾 Guardado opcional
    if save_summary:
        import pandas as pd
        pd.DataFrame(delot_version_info, columns=["Package", "Installed", "Required", "Status"]).to_csv(f"{summary_filename}.csv", index=False)
        with open(f"{summary_filename}.md", "w") as f:
            f.write("# 📋 Environment Report\n\n")
            f.write(delot_env_md + "\n\n" + delot_sys_md + "\n\n" + delot_md_table)

    # ⚠️ Reporte de fallos de import
    if delot_failed_imports:
        print("\n⚠️ Import errors:")
        for delot_mod, delot_msg in delot_failed_imports:
            print(f"  - {delot_mod}: {delot_msg}")

# 🚀 Ejecutar
delot_setup_environment(silent=True, save_summary=True)


## ⚙️ Environment Summary


**📍 Path:** `/Users/ludovicdelotbravo/Documents/S6 - LIT DELOT B. LUDOVIC/S6P1_AI_EMPRESARIAL/04_SCRIPTS`  
**🐍 Python:** `3.9.6`  
**💻 Platform:** `Darwin 24.3.0 (arm64)`  
**📦 Virtual Env:** `/Users/ludovicdelotbravo/Documents/S5 - LIT - DELOT B. LUDOVIC/S5P1_INTRO_ECONOMETRIA/04_SCRIPTS/.venv`  
**🧪 Test:** ✅ Functional test passed  
**⏱️ Time:** `1.67s`


### 🖥️ System Info
- **OS:** `Darwin 24.3.0 (arm64)`
- **Processor:** `Apple M1 Max`
- **Physical Cores:** `10`
- **Logical Cores:** `10`
- **RAM:** `68.72 GB`
- **Disk Free:** `104.67 GB`
- **Disk Total:** `1995.22 GB`
- **GPU:** `No GPU detected`

### 📋 Library Versions

| 📦 Package | 🔢 Installed | 🎯 Required | ✅ Status |
|------------|--------------|--------------|------------|
| `matplotlib` | `3.9.4` | `3.5.0` | ✅ |
| `numpy` | `2.0.2` | `1.22.0` | ✅ |
| `pandas` | `2.2.3` | `1.5.0` | ✅ |
| `plotly` | `6.0.0` | `5.5.0` | ✅ |
| `scikit-learn` | `1.6.1` | `1.0.0` | ✅ |
| `scipy` | `1.13.1` | `1.7.0` | ✅ |
| `seaborn` | `0.13.2` | `0.11.2` | ✅ |


# 2. Carga y Preprocesamiento de Datos

In [158]:
import pandas as pd

# Ruta del archivo
file_path = '../data/rfm_clientes_clusterizados_final.csv'  # NOTE: raw source export not included in repo; see README

# Cargar archivo CSV
try:
    df = pd.read_csv(file_path)
    print("✅ Archivo cargado correctamente.")
except Exception as e:
    print("❌ Error al leer el archivo:", e)
    raise

# Conversión de fechas
df['first_order_date'] = pd.to_datetime(df['first_order_date'], format='%d/%m/%y %H:%M')
df['last_order_date']  = pd.to_datetime(df['last_order_date'], format='%d/%m/%y %H:%M')

# Limpieza de columnas monetarias
df['average_order_value'] = df['average_order_value'].replace('[\$,]', '', regex=True).astype(float)
df['total_amount_spent']  = df['total_amount_spent'].replace('[\$,]', '', regex=True).astype(float)

# Cálculo de métricas RFM
current_date = pd.to_datetime("2024-09-02")
df['Recency']   = (current_date - df['last_order_date']).dt.days
df['Frequency'] = df['total_number_of_orders']
df['Monetary']  = df['total_amount_spent']

# ------------------ MINI EDA ------------------

print("\n📊 --- MINI EDA DEL DATAFRAME ---")

# Info general
print(f"\n➡️  Dimensiones del DataFrame: {df.shape[0]} filas, {df.shape[1]} columnas")
print("\n➡️  Tipos de datos:")
print(df.dtypes)

# Nulos
print("\n➡️  Valores nulos por columna:")
print(df.isnull().sum())

# Estadísticas básicas numéricas
print("\n➡️  Estadísticas descriptivas (numéricas):")
print(df.select_dtypes(include='number').describe().transpose())

# Estadísticas para columnas no numéricas (objetos y fechas)
print("\n➡️  Estadísticas descriptivas (categóricas / fechas):")
print(df.select_dtypes(include=['object', 'datetime']).describe().transpose())

# Valores únicos por columna (hasta 6 por cada una)
print("\n➡️  Valores únicos (máx 6) por columna:")
for col in df.columns:
    uniques = df[col].dropna().unique()
    sample_uniques = uniques[:6] if len(uniques) > 6 else uniques
    print(f"- {col} ({len(uniques)} únicos): {sample_uniques}")

# Preview del dataset
print("\n➡️  Primeras filas del DataFrame:")
print(df.head(3))

# RFM summary
print("\n📦 --- Resumen RFM ---")
print(df[['Client', 'Recency', 'Frequency', 'Monetary']].head())   

✅ Archivo cargado correctamente.

📊 --- MINI EDA DEL DATAFRAME ---

➡️  Dimensiones del DataFrame: 1412 filas, 10 columnas

➡️  Tipos de datos:
Client                              int64
accepts_email_marketing            object
first_order_date           datetime64[ns]
last_order_date            datetime64[ns]
total_number_of_orders              int64
average_order_value               float64
total_amount_spent                float64
Recency                             int64
Frequency                           int64
Monetary                          float64
dtype: object

➡️  Valores nulos por columna:
Client                       0
accepts_email_marketing    254
first_order_date             0
last_order_date              0
total_number_of_orders       0
average_order_value          0
total_amount_spent           0
Recency                      0
Frequency                    0
Monetary                     0
dtype: int64

➡️  Estadísticas descriptivas (numéricas):
                       

# 3. Ingeniería de Variables y Creación de la Tabla RFM

In [159]:
# 3. Ingeniería de Variables y Creación del DataFrame RFM Mejorado para Clustering
# ------------------------------------------------------------------------------
# Objetivos:
#   1. Filtrar registros con datos inconsistentes (p.ej.: Recency negativo).
#   2. Crear nuevos derivados que puedan mejorar el desempeño del clustering.
#   3. Trabajar con las columnas originales y sus transformaciones:
#       - total_number_of_orders --> Frecuencia
#       - total_amount_spent      --> Gasto (Monetary)
#   4. Calcular una variable derivada: Computed_AOV (Average Order Value)
#      como total_amount_spent / total_number_of_orders.
#   5. Crear una transformación logarítmica adicional para Recency.
#   6. Generar puntajes RFM (R_Score, F_Score y M_Score) usando cuartiles (pd.qcut).
#   7. Conformar un DataFrame final “rfm” que contenga las variables más relevantes
#      para la clusterización (incluyendo versiones logarítmicas de las variables).
#
# Nota: Revisa el mini EDA. Observamos que algunas filas tienen total_amount_spent = 0.0;
#       esto puede ser real o reflejar registros sin gasto. Aquí se conserva el valor, pero
#       se recomienda analizar el contexto de tus datos.

import numpy as np
import pandas as pd

# --- A) Filtrado de Registros con Recency Negativo ---
# Los clientes con Recency negativo indican que la última orden fue después
# de la fecha de referencia (2024-09-02), lo que puede ser un error.
rfm_full = df.copy()
initial_count = rfm_full.shape[0]
rfm_full = rfm_full[rfm_full['Recency'] >= 0]
print(f"\nFiltrado de Recency negativo: {initial_count - rfm_full.shape[0]} registros eliminados. Nuevo total: {rfm_full.shape[0]}.")

# --- B) Validación y Cálculo de Variables Derivadas ---
# Verificamos que 'total_amount_spent' se lea correctamente (ya lo comprobamos en el chunk 2).
print("\n📝 Verificación de 'total_amount_spent' (post-filtrado):")
print(rfm_full[['Client', 'total_amount_spent']].head(10))

# 1. Computed_AOV: Promedio de gasto por orden.
#    Se calcula como total_amount_spent / total_number_of_orders.
#    (Dado que total_number_of_orders tiene mínimo 2 según tu EDA, no habrá división por 0.)
rfm_full['Computed_AOV'] = rfm_full['total_amount_spent'] / rfm_full['total_number_of_orders']

# 2. Transformación logarítmica para Recency:
#    Se crea Log_Recency para reducir el sesgo de la variable Recency.
rfm_full['Log_Recency'] = np.log1p(rfm_full['Recency'])

# --- C) Creación de Variables Logarítmicas (ya existentes) para Frecuencia y Monetary ---
# Usamos np.log1p para suavizar la distribución de la frecuencia y el gasto.
rfm_full['Log_total_number_of_orders'] = np.log1p(rfm_full['total_number_of_orders'])
rfm_full['Log_total_amount_spent'] = np.log1p(rfm_full['total_amount_spent'])

# --- D) Selección de Variables Relevantes para Clustering ---
# Se conforma un DataFrame "rfm" que contendrá:
#   - Client
#   - Recency y Log_Recency
#   - total_number_of_orders y Log_total_number_of_orders
#   - total_amount_spent y Log_total_amount_spent
#   - Computed_AOV (puede aportar información adicional)
rfm = rfm_full[['Client', 'Recency', 'Log_Recency', 'total_number_of_orders', 'Log_total_number_of_orders',
                'total_amount_spent', 'Log_total_amount_spent', 'Computed_AOV']].copy()

# --- E) Generación de Puntajes RFM mediante Cuartiles ---
# Para efectos interpretativos, se calculan los puntajes:
#
# 1. R_Score (Recency):
#    Se ordena de menor a mayor porque un menor Recency (más reciente) es mejor.
rfm = rfm.sort_values('Recency')
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])

# 2. F_Score (Frecuencia):
#    Se utiliza 'total_number_of_orders'. Se usa ranking para resolver empates.
rfm = rfm.sort_values('total_number_of_orders')
rfm['F_Score'] = pd.qcut(rfm['total_number_of_orders'].rank(method="first"), 4, labels=[1, 2, 3, 4])

# 3. M_Score (Monetary):
#    Se utiliza 'total_amount_spent'. Se asignan cuartiles, donde a mayor gasto se le asigna un puntaje mayor.
rfm = rfm.sort_values('total_amount_spent')
rfm['M_Score'] = pd.qcut(rfm['total_amount_spent'], 4, labels=[1, 2, 3, 4])

# Se crea el RFM_Score concatenando los puntajes (útil para interpretación posterior).
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# --- F) Reinicio de Índices y Visualización Final ---
rfm = rfm.reset_index(drop=True)

print("\n📦 Ejemplo de Tabla RFM Mejorada:")
print(rfm.head(10))

# --- G) Resumen Estadístico de Variables Clave ---
print("\n🔎 Estadísticas de Variables para Clustering:")
print(rfm[['Recency', 'Log_Recency', 'total_number_of_orders', 'Log_total_number_of_orders',
           'total_amount_spent', 'Log_total_amount_spent', 'Computed_AOV']].describe())



Filtrado de Recency negativo: 29 registros eliminados. Nuevo total: 1383.

📝 Verificación de 'total_amount_spent' (post-filtrado):
    Client  total_amount_spent
0        1            85155.29
2        3            37912.65
3        4            35990.12
4        5            26753.75
5        6            26418.05
6        7            22903.30
7        8            17857.60
8       10            13787.15
9       11            13008.65
10      12            12704.55

📦 Ejemplo de Tabla RFM Mejorada:
   Client  Recency  Log_Recency  total_number_of_orders  \
0    6908      599     6.396930                       2   
1    6749      555     6.320768                       3   
2    7156      596     6.391917                       2   
3    7044      550     6.311735                       2   
4    6941      387     5.961005                       2   
5    6968      295     5.690359                       2   
6    7101      186     5.231109                       2   
7    7164      107   

In [160]:
# 4. Opciones de Clustering Avanzadas para Clientes
# -------------------------------------------------
# Este bloque implementa tres métodos de clustering sobre el DataFrame "rfm"
# (resultado del Chunk 3 avanzado) utilizando las siguientes variables:
#   - Log_Recency
#   - Log_total_number_of_orders
#   - Log_total_amount_spent
#   - Log_Computed_AOV
#
# Se escala el conjunto de variables con StandardScaler.
#
# Se aplican tres algoritmos:
#   Opción 1: KMeans (básico)
#   Opción 2: DBSCAN (basado en densidad, con postprocesado para evitar -1)
#   Opción 3: Gaussian Mixture Model (GMM, enfoque probabilístico)
#
# Al final, se ajustan las etiquetas para que comiencen en 1 (por ejemplo, 1, 2, 3, …).

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Asegurarnos de tener la variable Log_Computed_AOV, si no está, crearla.
if 'Log_Computed_AOV' not in rfm.columns:
    rfm['Log_Computed_AOV'] = np.log1p(rfm['Computed_AOV'])

# Selección de variables de entrada para el clustering.
features = ['Log_Recency', 'Log_total_number_of_orders', 'Log_total_amount_spent', 'Log_Computed_AOV']

# Escalar las variables para igualar sus escalas.
scaler = StandardScaler()
X = scaler.fit_transform(rfm[features])

# -------------------------------------------------
# Opción 1: KMeans (Básico)
# -------------------------------------------------
from sklearn.cluster import KMeans

# Usamos KMeans con 5 clusters (por ejemplo).
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
# Se sumará 1 para que las etiquetas comiencen en 1.
rfm['Cluster_KMeans'] = kmeans.fit_predict(X) + 1

# -------------------------------------------------
# Opción 2: DBSCAN (Basado en densidad)
# -------------------------------------------------
from sklearn.cluster import DBSCAN

# Definimos DBSCAN con parámetros que se puedan ajustar.
dbscan = DBSCAN(eps=0.5, min_samples=5)
raw_dbscan_labels = dbscan.fit_predict(X)

# Postprocesamiento para evitar valores -1:
#   - Si se detecta ruido (label = -1), se asigna a un nuevo cluster
#   - Luego se suma 1 a todas las etiquetas para que comiencen en 1.
if np.any(raw_dbscan_labels == -1):
    # Obtiene el mayor cluster válido (excluyendo -1)
    max_valid = np.max(raw_dbscan_labels[raw_dbscan_labels != -1])
    # Asigna el nuevo cluster para el ruido como max_valid + 1
    raw_dbscan_labels = np.where(raw_dbscan_labels == -1, max_valid + 1, raw_dbscan_labels)
# Ajustar para que comiencen en 1:
dbscan_labels = raw_dbscan_labels + 1
rfm['Cluster_DBSCAN'] = dbscan_labels

# -------------------------------------------------
# Opción 3: Gaussian Mixture Model (GMM)
# -------------------------------------------------
from sklearn.mixture import GaussianMixture

# Usamos GMM con 5 componentes (similar a KMeans) como ejemplo.
gmm = GaussianMixture(n_components=5, random_state=42)
# Suma 1 para etiquetas que inicien en 1.
rfm['Cluster_GMM'] = gmm.fit_predict(X) + 1

# -------------------------------------------------
# Guardar DataFrames de Resultados (opcional)
df_kmeans = rfm.copy()  # Contendrá la columna 'Cluster_KMeans'
df_dbscan  = rfm.copy()  # Contendrá la columna 'Cluster_DBSCAN'
df_gmm     = rfm.copy()  # Contendrá la columna 'Cluster_GMM'

# Ejemplos preliminares de asignaciones (sin gráficos)
print("\nEjemplo de clusters con KMeans:")
print(rfm[['Client', 'Cluster_KMeans']].head(10))
print("\nEjemplo de clusters con DBSCAN:")
print(rfm[['Client', 'Cluster_DBSCAN']].head(10))
print("\nEjemplo de clusters con GMM:")
print(rfm[['Client', 'Cluster_GMM']].head(10))



Ejemplo de clusters con KMeans:
   Client  Cluster_KMeans
0    6908               3
1    6749               3
2    7156               3
3    7044               3
4    6941               3
5    6968               3
6    7101               3
7    7164               3
8    6957               3
9    6923               3

Ejemplo de clusters con DBSCAN:
   Client  Cluster_DBSCAN
0    6908               1
1    6749               8
2    7156               1
3    7044               1
4    6941               1
5    6968               1
6    7101               1
7    7164               1
8    6957               8
9    6923               8

Ejemplo de clusters con GMM:
   Client  Cluster_GMM
0    6908            3
1    6749            3
2    7156            3
3    7044            3
4    6941            3
5    6968            3
6    7101            3
7    7164            3
8    6957            3
9    6923            3


In [161]:
# 5. Visualizaciones Avanzadas: Separación por Algoritmo y Comparación
# --------------------------------------------------------------------

import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_samples, silhouette_score
import numpy as np
import pandas as pd

# Se asume que 'rfm' es el DataFrame resultante del Chunk 4 que ya contiene:
# - Cluster_KMeans
# - Cluster_DBSCAN
# - Cluster_GMM
# Y las variables clave transformadas:
#   Log_Recency, Log_total_number_of_orders, Log_total_amount_spent, Log_Computed_AOV

# ====================================================================
# SECCIÓN A: Visualizaciones Separadas por Algoritmo
# ====================================================================

print("\n--- SECCIÓN A: Análisis Individual de Cada Algoritmo ---\n")

# --------------------------------
# 1. Gráficos 3D Interactivos para cada algoritmo
# --------------------------------
# KMeans 3D
fig_kmeans = px.scatter_3d(
    rfm,
    x="Log_Recency",
    y="Log_total_number_of_orders",
    z="Log_total_amount_spent",
    color="Cluster_KMeans",
    title="3D Scatter Plot - KMeans Clusters",
    labels={"Log_Recency": "Log(Recency)",
            "Log_total_number_of_orders": "Log(Frecuencia)",
            "Log_total_amount_spent": "Log(Gasto Total)"}
)
fig_kmeans.show()
print("Interpretación KMeans 3D: Se observa la agrupación de clientes basada en sus dimensiones logarítmicas. Grupos bien formados indican una segmentación coherente.")

# DBSCAN 3D
fig_dbscan = px.scatter_3d(
    rfm,
    x="Log_Recency",
    y="Log_total_number_of_orders",
    z="Log_total_amount_spent",
    color="Cluster_DBSCAN",
    title="3D Scatter Plot - DBSCAN Clusters",
    labels={"Log_Recency": "Log(Recency)",
            "Log_total_number_of_orders": "Log(Frecuencia)",
            "Log_total_amount_spent": "Log(Gasto Total)"}
)
fig_dbscan.show()
print("Interpretación DBSCAN 3D: Permite visualizar cómo DBSCAN agrupa clientes por densidad, resaltando posibles puntos de ruido que se han reasignado.")

# GMM 3D
fig_gmm = px.scatter_3d(
    rfm,
    x="Log_Recency",
    y="Log_total_number_of_orders",
    z="Log_total_amount_spent",
    color="Cluster_GMM",
    title="3D Scatter Plot - GMM Clusters",
    labels={"Log_Recency": "Log(Recency)",
            "Log_total_number_of_orders": "Log(Frecuencia)",
            "Log_total_amount_spent": "Log(Gasto Total)"}
)
fig_gmm.show()
print("Interpretación GMM 3D: El modelo probabilístico (GMM) puede mostrar solapamientos entre clusters o separaciones claras, lo que es visible en esta gráfica.")

# --------------------------------
# 2. Visualización 2D con t-SNE para cada algoritmo
# --------------------------------
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(
    scaler.transform(rfm[["Log_Recency", "Log_total_number_of_orders", "Log_total_amount_spent", "Log_Computed_AOV"]])
)
rfm["TSNE_1"] = X_tsne[:, 0]
rfm["TSNE_2"] = X_tsne[:, 1]

# t-SNE para KMeans
fig_tsne_kmeans = px.scatter(
    rfm,
    x="TSNE_1",
    y="TSNE_2",
    color="Cluster_KMeans",
    title="t-SNE 2D Scatter Plot - KMeans",
    labels={"TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"}
)
fig_tsne_kmeans.show()
print("Interpretación t-SNE KMeans: La proyección 2D permite visualizar la separación entre los clusters obtenidos con KMeans.")

# t-SNE para DBSCAN
fig_tsne_dbscan = px.scatter(
    rfm,
    x="TSNE_1",
    y="TSNE_2",
    color="Cluster_DBSCAN",
    title="t-SNE 2D Scatter Plot - DBSCAN",
    labels={"TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"}
)
fig_tsne_dbscan.show()
print("Interpretación t-SNE DBSCAN: Se evidencia cómo DBSCAN ha identificado agrupaciones basadas en densidad, con posibles clusters de ruido reasignados.")

# t-SNE para GMM
fig_tsne_gmm = px.scatter(
    rfm,
    x="TSNE_1",
    y="TSNE_2",
    color="Cluster_GMM",
    title="t-SNE 2D Scatter Plot - GMM",
    labels={"TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"}
)
fig_tsne_gmm.show()
print("Interpretación t-SNE GMM: La visualización 2D de GMM permite apreciar la distribución probabilística de los clusters y posibles solapamientos.")

# --------------------------------
# 3. Análisis de Silhouette (ejemplo con KMeans)
# --------------------------------
labels_kmeans = rfm['Cluster_KMeans'].values - 1  # Para análisis 0-index
sil_score = silhouette_score(X, labels_kmeans)
print(f"\nSilhouette Score para KMeans: {sil_score:.3f}")
sample_sil_values = silhouette_samples(X, labels_kmeans)
rfm["Silhouette"] = sample_sil_values

# Graficar el silhouette plot para KMeans
y_lower = 10
n_clusters = len(np.unique(labels_kmeans))
plt.figure(figsize=(10, 6))
for i in range(n_clusters):
    ith_cluster_sil_values = sample_sil_values[labels_kmeans == i]
    ith_cluster_sil_values.sort()
    size_cluster_i = ith_cluster_sil_values.shape[0]
    y_upper = y_lower + size_cluster_i
    color = plt.cm.nipy_spectral(float(i) / n_clusters)
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil_values,
                      facecolor=color, edgecolor=color, alpha=0.7)
    plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i+1))
    y_lower = y_upper + 10  # Espacio entre clusters

plt.title("Silhouette Plot - KMeans")
plt.xlabel("Coeficiente de Silhouette")
plt.ylabel("Etiqueta de Cluster")
plt.axvline(x=sil_score, color="red", linestyle="--")
plt.yticks([])
plt.show()
print("Interpretación Silhouette (KMeans): Valores altos y barras largas sugieren clusters bien definidos; la línea roja indica el promedio global.")

# --------------------------------
# 4. Tablas Resumen y Análisis de Distribuciones por Algoritmo
# --------------------------------
summary_vars = ["Recency", "total_number_of_orders", "total_amount_spent", "Computed_AOV"]

# Tablas resumen
print("\nResumen de clusters - KMeans:")
summary_kmeans = rfm.groupby("Cluster_KMeans")[summary_vars].agg(["min", "max", "mean", "std"])
print(summary_kmeans.round(2))
print("Interpretación: Las estadísticas resumen ayudan a distinguir el perfil de clientes por cada cluster.")

print("\nResumen de clusters - DBSCAN:")
summary_dbscan = rfm.groupby("Cluster_DBSCAN")[summary_vars].agg(["min", "max", "mean", "std"])
print(summary_dbscan.round(2))
print("Interpretación: Permite observar cómo varían los comportamientos en función de la densidad identificada por DBSCAN.")

print("\nResumen de clusters - GMM:")
summary_gmm = rfm.groupby("Cluster_GMM")[summary_vars].agg(["min", "max", "mean", "std"])
print(summary_gmm.round(2))
print("Interpretación: Muestra el perfil promedio y la dispersión de las variables en cada segmento GMM.")

# Gráficos de distribución: Boxplots y Countplots
plt.figure(figsize=(8, 5))
sns.boxplot(x="Cluster_KMeans", y="total_amount_spent", data=rfm, palette="Set3")
plt.title("Boxplot de Gasto Total por Cluster (KMeans)")
plt.xlabel("Cluster KMeans")
plt.ylabel("total_amount_spent")
plt.show()
print("Interpretación Boxplot KMeans: Se observa la variabilidad en el gasto total entre clusters.")

plt.figure(figsize=(8, 5))
sns.boxplot(x="Cluster_GMM", y="Recency", data=rfm, palette="Set2")
plt.title("Boxplot de Recency por Cluster (GMM)")
plt.xlabel("Cluster GMM")
plt.ylabel("Recency (días)")
plt.show()
print("Interpretación Boxplot GMM: Indica diferencias en la antigüedad de la última compra entre clusters.")

plt.figure(figsize=(8, 5))
sns.countplot(x="Cluster_KMeans", data=rfm, palette="viridis")
plt.title("Cantidad de Clientes por Cluster (KMeans)")
plt.xlabel("Cluster KMeans")
plt.ylabel("Número de Clientes")
plt.show()
print("Interpretación Countplot KMeans: Visualiza el balance de clientes asignados a cada cluster.")

# -----------------------------------------------------------------------------------
# SECCIÓN B: Visualizaciones Comparativas Entre Algoritmos
# -----------------------------------------------------------------------------------
print("\n--- SECCIÓN B: Comparación entre Algoritmos ---\n")

# Comparación de Distribuciones: Distribución de 'total_amount_spent' por cluster de cada algoritmo
plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
sns.boxplot(x="Cluster_KMeans", y="total_amount_spent", data=rfm, palette="Set3")
plt.title("Gasto Total por Cluster (KMeans)")
plt.xlabel("KMeans")
plt.ylabel("total_amount_spent")

plt.subplot(1, 3, 2)
sns.boxplot(x="Cluster_DBSCAN", y="total_amount_spent", data=rfm, palette="Set3")
plt.title("Gasto Total por Cluster (DBSCAN)")
plt.xlabel("DBSCAN")
plt.ylabel("")  # No repetir etiqueta

plt.subplot(1, 3, 3)
sns.boxplot(x="Cluster_GMM", y="total_amount_spent", data=rfm, palette="Set3")
plt.title("Gasto Total por Cluster (GMM)")
plt.xlabel("GMM")
plt.tight_layout()
plt.show()
print("Interpretación Comparativa (Boxplots): Se comparan las distribuciones del gasto total según el cluster asignado por cada algoritmo, facilitando la identificación de segmentos de alto y bajo gasto entre métodos.")

# Comparación de Distribución de 'Recency'
plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
sns.boxplot(x="Cluster_KMeans", y="Recency", data=rfm, palette="Set2")
plt.title("Recency por Cluster (KMeans)")
plt.xlabel("KMeans")
plt.ylabel("Recency")

plt.subplot(1, 3, 2)
sns.boxplot(x="Cluster_DBSCAN", y="Recency", data=rfm, palette="Set2")
plt.title("Recency por Cluster (DBSCAN)")
plt.xlabel("DBSCAN")
plt.ylabel("")

plt.subplot(1, 3, 3)
sns.boxplot(x="Cluster_GMM", y="Recency", data=rfm, palette="Set2")
plt.title("Recency por Cluster (GMM)")
plt.xlabel("GMM")
plt.tight_layout()
plt.show()
print("Interpretación Comparativa (Recency): Se observa cómo difiere la antigüedad de la última compra en cada cluster, permitiendo evaluar la vitalidad de los segmentos según cada método.")

# También podemos comparar distribuciones de Computed_AOV (valor promedio por orden)
plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
sns.boxplot(x="Cluster_KMeans", y="Computed_AOV", data=rfm, palette="Set1")
plt.title("AOV por Cluster (KMeans)")
plt.xlabel("KMeans")
plt.ylabel("Computed_AOV")

plt.subplot(1, 3, 2)
sns.boxplot(x="Cluster_DBSCAN", y="Computed_AOV", data=rfm, palette="Set1")
plt.title("AOV por Cluster (DBSCAN)")
plt.xlabel("DBSCAN")
plt.ylabel("")

plt.subplot(1, 3, 3)
sns.boxplot(x="Cluster_GMM", y="Computed_AOV", data=rfm, palette="Set1")
plt.title("AOV por Cluster (GMM)")
plt.xlabel("GMM")
plt.tight_layout()
plt.show()
print("Interpretación Comparativa (AOV): Este gráfico permite comparar el valor promedio por orden en cada cluster, lo que puede ayudar a identificar clientes de alto valor versus clientes con compras de bajo impacto.")

# -----------------------------------------------------------------------------------
# F) Comentario Final
# -----------------------------------------------------------------------------------
print("\nInterpretación Global: Al observar las visualizaciones individuales y comparativas, se puede apreciar que cada algoritmo genera diferentes segmentaciones basadas en la estructura de los datos. \n" +
      "• KMeans tiende a generar clusters con formas bien definidas y equilibradas.\n" +
      "• DBSCAN puede identificar clusters de densidad y, dependiendo de los parámetros, reubicar los puntos de ruido en un cluster adicional.\n" +
      "• GMM, al ser probabilístico, puede evidenciar solapamientos entre clusters o diferencias claras según la heterogeneidad de los datos.\n" +
      "Las gráficas comparativas ayudan a contrastar las diferencias en la variable 'total_amount_spent', 'Recency' y 'Computed_AOV' entre los métodos, permitiéndote tomar decisiones informadas sobre qué segmentación se adapta mejor al objetivo de negocio.")



--- SECCIÓN A: Análisis Individual de Cada Algoritmo ---



Interpretación KMeans 3D: Se observa la agrupación de clientes basada en sus dimensiones logarítmicas. Grupos bien formados indican una segmentación coherente.


Interpretación DBSCAN 3D: Permite visualizar cómo DBSCAN agrupa clientes por densidad, resaltando posibles puntos de ruido que se han reasignado.


Interpretación GMM 3D: El modelo probabilístico (GMM) puede mostrar solapamientos entre clusters o separaciones claras, lo que es visible en esta gráfica.


Interpretación t-SNE KMeans: La proyección 2D permite visualizar la separación entre los clusters obtenidos con KMeans.


...

Interpretación Comparativa (Boxplots): Se comparan las distribuciones del gasto total según el cluster asignado por cada algoritmo, facilitando la identificación de segmentos de alto y bajo gasto entre métodos.


/var/folders/33/y5pqbn8s76b1k47km33_cxc40000gn/T/ipykernel_69158/1120239120.py:228: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


/var/folders/33/y5pqbn8s76b1k47km33_cxc40000gn/T/ipykernel_69158/1120239120.py:234: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


/var/folders/33/y5pqbn8s76b1k47km33_cxc40000gn/T/ipykernel_69158/1120239120.py:240: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




Interpretación Comparativa (Recency): Se observa cómo difiere la antigüedad de la última compra en cada cluster, permitiendo evaluar la vitalidad de los segmentos según cada método.


/var/folders/33/y5pqbn8s76b1k47km33_cxc40000gn/T/ipykernel_69158/1120239120.py:250: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


/var/folders/33/y5pqbn8s76b1k47km33_cxc40000gn/T/ipykernel_69158/1120239120.py:256: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


/var/folders/33/y5pqbn8s76b1k47km33_cxc40000gn/T/ipykernel_69158/1120239120.py:262: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




Interpretación Comparativa (AOV): Este gráfico permite comparar el valor promedio por orden en cada cluster, lo que puede ayudar a identificar clientes de alto valor versus clientes con compras de bajo impacto.

Interpretación Global: Al observar las visualizaciones individuales y comparativas, se puede apreciar que cada algoritmo genera diferentes segmentaciones basadas en la estructura de los datos. 
• KMeans tiende a generar clusters con formas bien definidas y equilibradas.
• DBSCAN puede identificar clusters de densidad y, dependiendo de los parámetros, reubicar los puntos de ruido en un cluster adicional.
• GMM, al ser probabilístico, puede evidenciar solapamientos entre clusters o diferencias claras según la heterogeneidad de los datos.
Las gráficas comparativas ayudan a contrastar las diferencias en la variable 'total_amount_spent', 'Recency' y 'Computed_AOV' entre los métodos, permitiéndote tomar decisiones informadas sobre qué segmentación se adapta mejor al objetivo de negoc

In [162]:
import os
import openai
import pandas as pd

# Configurar la clave API desde la variable de entorno.
# Reemplaza la cadena predeterminada con tu clave en caso de pruebas (no lo hagas en producción).
openai.api_key = os.getenv("OPENAI_API_KEY")  # requires .env with OPENAI_API_KEY

# Instancia un cliente (esto es recomendado en vez de usar llamadas globales)
client = openai.OpenAI(api_key=openai.api_key)

def chain_prompt(prompt: str, prior_context: str = "") -> str:
    """
    Envía un prompt a GPT-4 utilizando el output previo como contexto y retorna la respuesta.
    """
    combined_prompt = f"{prior_context}\n\n{prompt}" if prior_context else prompt
    
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": (
                    "Eres un científico de datos con doctorado y amplia experiencia en análisis de clusters e ingeniería de prompts. "
                    "Proporciona respuestas estructuradas y detalladas." 
                )},
                {"role": "user", "content": combined_prompt}
            ],
            temperature=0.7,
            max_tokens=1500
        )
    except Exception as e:
        print("Error al llamar a la API de OpenAI:", e)
        raise
    return response.choices[0].message.content

# ------------------------------------------------------------------------------
# Se asume que 'rfm' es el DataFrame final procesado que contiene las siguientes columnas:
# - "Recency", "total_number_of_orders", "total_amount_spent", "Computed_AOV"
# - "Log_Recency", "Log_total_number_of_orders", "Log_total_amount_spent", "Log_Computed_AOV"
# - Clusters asignados: 'Cluster_KMeans', 'Cluster_DBSCAN', 'Cluster_GMM'
# ------------------------------------------------------------------------------

# Extraer resúmenes estadísticos (convertidos a string) de cada modelo
kmeans_summary = rfm.groupby('Cluster_KMeans')[['Recency', 'total_number_of_orders', 'total_amount_spent', 'Computed_AOV']]\
    .agg(["min", "max", "mean", "std"]).round(2).to_string()

dbscan_summary = rfm.groupby('Cluster_DBSCAN')[['Recency', 'total_number_of_orders', 'total_amount_spent', 'Computed_AOV']]\
    .agg(["min", "max", "mean", "std"]).round(2).to_string()

gmm_summary = rfm.groupby('Cluster_GMM')[['Recency', 'total_number_of_orders', 'total_amount_spent', 'Computed_AOV']]\
    .agg(["min", "max", "mean", "std"]).round(2).to_string()

# Valores de métricas globales (modifica estos valores con los reales obtenidos en tu análisis)
sil_score_value = 0.45         # Silhouette Score de KMeans (ejemplo)
davies_bouldin_value = 2.8     # Davies-Bouldin Index de KMeans (ejemplo)
calinski_harabasz_value = 1500 # Calinski-Harabasz Index de KMeans (ejemplo)

metrics_info = (
    f"**Métricas Globales para KMeans:**\n"
    f"- Silhouette Score: {sil_score_value:.3f}\n"
    f"- Davies-Bouldin Index: {davies_bouldin_value:.3f}\n"
    f"- Calinski-Harabasz Index: {calinski_harabasz_value:.3f}\n\n"
    f"**Resumen de Clusters - KMeans:**\n```\n{kmeans_summary}\n```\n\n"
    f"**Resumen de Clusters - DBSCAN:**\n```\n{dbscan_summary}\n```\n\n"
    f"**Resumen de Clusters - GMM:**\n```\n{gmm_summary}\n```"
)

# ------------------------------------------------------------------------------
# Construcción de Prompts Encadenados
# ------------------------------------------------------------------------------

# 1. Análisis de KMeans
prompt_kmeans = (
    "Analiza en profundidad los resultados del modelo KMeans aplicado a la segmentación de clientes. "
    "Utiliza la siguiente información:\n\n" + metrics_info + "\n\n"
    "Explica detalladamente cómo las métricas (Silhouette Score, Davies-Bouldin Index y Calinski-Harabasz Index) "
    "reflejan la cohesión interna y separación entre clusters. Describe el perfil de cada cluster según los resúmenes "
    "estadísticos de 'Recency', 'total_number_of_orders', 'total_amount_spent' y 'Computed_AOV', identificando segmentos "
    "como clientes de alto valor, clientes frecuentes o inactivos. Concluye resumiendo fortalezas y limitaciones de KMeans."
)
result_kmeans = chain_prompt(prompt_kmeans)
print("=== Análisis KMeans ===\n")
print(result_kmeans)

# 2. Análisis de DBSCAN (encadenado con output de KMeans)
prompt_dbscan = (
    "Utilizando el análisis de KMeans proporcionado anteriormente como contexto, analiza el modelo DBSCAN aplicado a la segmentación de clientes. "
    "Detalla cómo DBSCAN identifica clusters basados en la densidad y maneja puntos de ruido (aquellos originalmente etiquetados como -1, luego reasignados). "
    "Interpreta el resumen estadístico de los clusters obtenidos con DBSCAN y compáralo brevemente con KMeans, "
    "destacando sus ventajas para detectar formas arbitrarias y gestionar outliers, además de sus limitaciones."
)
result_dbscan = chain_prompt(prompt_dbscan, prior_context=result_kmeans)
print("\n=== Análisis DBSCAN ===\n")
print(result_dbscan)

# 3. Análisis de GMM (encadenado con output de DBSCAN)
prompt_gmm = (
    "Basándote en los análisis previos de KMeans y DBSCAN, analiza el modelo Gaussian Mixture Model (GMM) aplicado a la segmentación de clientes. "
    "Describe cómo GMM utiliza distribuciones de probabilidad para modelar la incertidumbre en la asignación de clusters y capturar la heterogeneidad en el comportamiento de los clientes. "
    "Interpreta detalladamente el resumen estadístico de los clusters obtenidos con GMM y compáralo con los resultados de KMeans y DBSCAN, "
    "destacando la capacidad de GMM para captar solapamientos y diferencias sutiles entre los perfiles de los clientes."
)
result_gmm = chain_prompt(prompt_gmm, prior_context=result_dbscan)
print("\n=== Análisis GMM ===\n")
print(result_gmm)

# 4. Análisis Comparativo Global y Recomendaciones (encadenado con output de GMM)
prompt_comparative = (
    "Finalmente, realiza un análisis comparativo global entre los tres métodos de segmentación: KMeans, DBSCAN y GMM. "
    "Utiliza toda la información y análisis previos para comparar la cohesión interna, la separación de clusters y la capacidad de capturar "
    "la heterogeneidad en el comportamiento de los clientes. Discute las ventajas y desventajas de cada método en términos de manejo de outliers, "
    "detección de formas arbitrarias y robustez. Concluye ofreciendo recomendaciones fundamentadas sobre cuál modelo es el más adecuado para una estrategia "
    "de inteligencia de negocios o si combinar los resultados de varios métodos podría mejorar la segmentación."
)
result_comparative = chain_prompt(prompt_comparative, prior_context=result_gmm)
print("\n=== Análisis Comparativo Global ===\n")
print(result_comparative)


=== Análisis KMeans ===

Las métricas proporcionadas brindan información valiosa sobre la calidad de los clusters generados por el modelo KMeans.

El Silhouette Score oscila entre -1 y 1. Un valor alto indica que los puntos de datos se agrupan correctamente en los clusters correctos, y están lejos de los límites de decisión entre clusters. Un valor de 0.45 indica que los clusters están razonablemente definidos y separados, aunque todavía hay margen para mejorar el agrupamiento.

El Davies-Bouldin Index mide la similitud media entre los clusters. Cuanto menor sea este valor, mejor, ya que indica que los clusters están más separados y menos dispersos. Un valor de 2.8 es relativamente alto, lo que sugiere que hay una superposición significativa entre los clusters.

El Calinski-Harabasz Index mide la dispersión entre clusters y la dispersión dentro de los clusters. Un valor más alto es mejor, ya que indica que los clusters están bien separados. Un valor de 1500 es relativamente alto, indic

In [181]:
import os
import re
import json
import openai
import pandas as pd
from IPython.display import display, Markdown

# ----------------------------------------------------------------------------
# 1) Configuración de la API (con tu clave directamente)
# ----------------------------------------------------------------------------
openai.api_key = os.getenv("OPENAI_API_KEY")  # requires .env with OPENAI_API_KEY
client = openai.OpenAI(api_key=openai.api_key)

# ----------------------------------------------------------------------------
# 2) Few-shot + función robusta
# ----------------------------------------------------------------------------
FEW_SHOT_EXAMPLE = """
### Ejemplo de salida esperada (JSON)
```json
{
  "nombre": "Compradores Premium Recientes",
  "descripcion": "Clientes con alta frecuencia (media 8.5 compras) y gasto elevado (media $12,000). Recency baja (media 30 días), ...",
  "recomendaciones": [
    "🎯 Lanzar un programa VIP con beneficios exclusivos",
    "📧 Email marketing semanal con ofertas premium",
    "🔄 Upsell de productos complementarios de alta gama"
  ]
}
"""
    
def chain_prompt_structured(role: str, task: str, context: str, instructions: str) -> dict:
    """ Envía un prompt estructurado a GPT y extrae de forma robusta el JSON """
    prompt = f"""[ROL] {role}

[TAREA] {task}

[CONTEXTO] {context}

[EJEMPLO] {FEW_SHOT_EXAMPLE}

[INSTRUCCIONES] {instructions}

Pasos internos (piensa paso a paso) y al final haz una self‑critique breve de tu respuesta."""

    resp = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "Eres un científico de datos experto en análisis de clusters y prompt engineering."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=1200
    )

    text = resp.choices[0].message.content.strip()

    # 1. Intenta extraer bloque entre ```json ... ```
    json_block = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if json_block:
        try:
            return json.loads(json_block.group(1))
        except json.JSONDecodeError:
            pass

    # 2. Fallback: primer diccionario del texto
    fallback_block = re.search(r"(\{.*\})", text, re.DOTALL)
    if fallback_block:
        try:
            return json.loads(fallback_block.group(1))
        except json.JSONDecodeError as e:
            raise ValueError(f"Error al parsear JSON (fallback): {e}\nBloque:\n{fallback_block.group(1)}")

    raise ValueError(f"No se encontró un bloque JSON válido en la respuesta:\n{text}")

# ----------------------------------------------------------------------------
# 3) Generar perfil por cluster
# ----------------------------------------------------------------------------
def generate_cluster_profile(df: pd.DataFrame, cluster_column: str, cluster_value) -> dict:
    sub = df[df[cluster_column] == cluster_value]
    stats = sub[['Recency','total_number_of_orders','total_amount_spent','Computed_AOV']].describe().round(2).to_dict()
    context = json.dumps(stats, indent=2, ensure_ascii=False)

    role = "Actúa como un científico de datos con PhD especializado en análisis de clusters."
    task = f"Genera el perfil del cluster {cluster_value}."
    instructions = (
        "1. Devuelve un JSON con claves: nombre (string), descripcion (string) y recomendaciones (lista de strings).\n"
        "2. El nombre debe ser breve y descriptivo.\n"
        "3. La descripción debe analizar recency, frecuencia y gasto, citando estadísticas.\n"
        "4. Incluye al menos 3 recomendaciones estratégicas con emojis.\n"
        "5. Sigue el formato del ejemplo EXACTAMENTE."
    )

    return chain_prompt_structured(role, task, context, instructions)

# ----------------------------------------------------------------------------
# 4) Generar todos los perfiles y mostrarlos
# ----------------------------------------------------------------------------
cluster_profiles = {}
for algo, col in {
    "KMeans": "Cluster_KMeans",
    "DBSCAN": "Cluster_DBSCAN",
    "GMM": "Cluster_GMM"
}.items():
    display(Markdown(f"## 📊 Perfiles Modelo {algo}"))
    cluster_profiles[algo] = {}
    for cl in sorted(rfm[col].unique()):
        profile = generate_cluster_profile(rfm, col, cl)
        cluster_profiles[algo][cl] = profile
        display(Markdown(f"### Cluster {cl}"))
        display(Markdown(f"**Nombre:** {profile['nombre']}"))
        display(Markdown(f"**Descripción:** {profile['descripcion']}"))
        display(Markdown("**Recomendaciones:**"))
        for rec in profile['recomendaciones']:
            display(Markdown(f"- {rec}"))
    display(Markdown("---"))

# ----------------------------------------------------------------------------
# 5) Tabla comparativa
# ----------------------------------------------------------------------------
rows = []
for algo, profs in cluster_profiles.items():
    for cl, p in profs.items():
        rows.append({
            "🔸 Algoritmo": algo,
            "🔢 Cluster": cl,
            "🏷️ Nombre": p["nombre"],
            "📝 Descripción": p["descripcion"],
            "💡 Recomendaciones": "; ".join(p["recomendaciones"])
        })

df_comp = pd.DataFrame(rows)
display(Markdown("## 📋 Comparativa Global de Perfiles"))
display(df_comp.style
    .set_properties(subset=["📝 Descripción"], **{"width": "400px"})
    .set_properties(subset=["💡 Recomendaciones"], **{"width": "300px"})
    .hide(axis="index"))




## 📊 Perfiles Modelo KMeans

### Cluster 1

**Nombre:** Compradores Recientes de Alto Valor

**Descripción:** Estos clientes han realizado un promedio de 7.27 compras y han gastado en promedio $10,112.11. Han realizado una compra recientemente, con una media de 71.39 días desde la última compra. El valor medio del pedido es de $1,316.3, lo que indica que suelen comprar productos de mayor valor.

**Recomendaciones:**

...

- 📧 Enviar actualizaciones por correo electrónico sobre nuevos productos de alto valor

- 💼 Ofrecer un programa de lealtad premium para recompensar el gasto alto

---

## 📋 Comparativa Global de Perfiles

🔸 Algoritmo,🔢 Cluster,🏷️ Nombre,📝 Descripción,💡 Recomendaciones
KMeans,1,Compradores Recientes de Alto Valor,"Estos clientes han realizado un promedio de 7.27 compras y han gastado en promedio $10,112.11. Han realizado una compra recientemente, con una media de 71.39 días desde la última compra. El valor medio del pedido es de $1,316.3, lo que indica que suelen comprar productos de mayor valor.",💎 Ofrecer productos exclusivos o de edición limitada para mantener su interés y alentar compras de alto valor; 📲 Enviar recordatorios personalizados y ofertas exclusivas a través de notificaciones push o mensajes de texto; 🔄 Implementar un programa de recompensas para alentar repetidas compras
KMeans,2,Compradores Esporádicos de Alta Inversión,"Clientes que realizan compras con poca frecuencia (media de 2 compras), pero cuando compran, su gasto es significativo (media de $1867.76). Su recency es alta (media de 270 días), lo que indica que no han comprado recientemente.",🎁 Ofrecer incentivos o descuentos atractivos para fomentar compras más frecuentes.; 📧 Enviar recordatorios de correo electrónico periódicos con productos relevantes y ofertas especiales.; 💳 Implementar un programa de lealtad para recompensar sus compras de alto valor.
KMeans,3,Compradores Inactivos con Órdenes Múltiples,"Este cluster está compuesto por clientes que no han realizado compras recientemente (recency media de 386.92 días) pero que han realizado múltiples pedidos en el pasado (media de 2.31 órdenes). Sin embargo, no han generado ingresos en el período de tiempo considerado (media del total gastado y AOV de $0).",🏷️ Ofrecer descuentos para incentivar a estos clientes a realizar una nueva compra; 👥 Implementar programas de reactivación de clientes para aumentar su compromiso; 📧 Enviar correos electrónicos personalizados recordándoles sus productos favoritos en el pasado
KMeans,4,Compradores Infrecuentes pero de Alta Gama,"Clientes que no compran con frecuencia (media de 3.44 compras) pero cuando lo hacen, gastan bastante (media de $3176.03). La recency es relativamente alta (media de 248.84 días), lo que sugiere que estos clientes no han comprado recientemente. El valor medio de pedido es de $936.12, lo cual es considerablemente alto.",💡 Desarrollar un programa de lealtad para incentivar compras más frecuentes; 🎁 Ofrecer incentivos o descuentos en compras de alto valor para atraer a estos clientes a comprar más a menudo; 📧 Enviar correos electrónicos con productos de alta gama y ofertas especiales
KMeans,5,Clientes Leales con Gasto Moderado,"Este cluster agrupa a clientes con una frecuencia de compra moderada (media de 2.65 compras), que han realizado una compra recientemente (media de 31.2 días). El monto total gastado por estos clientes es relativamente alto (media de $2887.14), con un Valor Medio del Pedido (AOV) de alrededor de $1101.14. La variabilidad en el gasto es considerable, indicado por una desviación estándar alta en el monto total gastado y en el AOV.",🎁 Ofrezca incentivos de lealtad o recompensas para mantener su frecuencia de compra; 💌 Envíe recordatorios de compra a los 30 días para mantener su recency; 💰 Promocione productos de precio medio para aumentar el Valor Medio del Pedido (AOV)
DBSCAN,1,Clientes Inactivos Recientes,"Este cluster se caracteriza por clientes que han realizado compras recientemente (media de 389.98 días). Sin embargo, han realizado un número mínimo de compras (media de 2 compras) y no han gastado nada en total (media de gasto total de $0).",📧 Lanzar una campaña de email marketing para motivar compras recurrentes.; 💼 Ofrecer descuentos en sus próximas compras para incentivar el gasto.; 🔄 Implementar un programa de fidelidad para motivar la repetición de compra.
DBSCAN,2,Compradores Ocasionales de Alto Valor,"Este grupo de clientes realiza compras con poca frecuencia (media de 2 compras), pero con un alto gasto promedio (media de $1892.04). Su recency es relativamente alta (media de 250.29 días), lo que sugiere que pueden h